# 07 · Intermediate · Lakeflow jobs

Two checks on your Lakeflow Jobs, read from `system.lakeflow.jobs` and
`system.lakeflow.job_run_timeline` (or your organization's views of them):

| Check | Flags a job when |
|---|---|
| `jobs.never_run` | it is at least `never_run_days` old and no run is observed |
| `jobs.failing_streak` | every finished run failed on each of the last `failing_days` days |

Both give `review` only; nothing is paused or deleted. Run history goes back about a
year, so "no run observed" never means "never ran". When `create_time` is empty (older
rows), a job's age comes from its earliest recorded change and is shown as "about N days".

In [ ]:
%pip install /Volumes/main/default/lakehouse_gc/lakehouse_gc-0.4.0-py3-none-any.whl

In [ ]:
from lakehouse_gc import check_access, rules_dataframe, run

POLICY_PATH = "/Volumes/main/default/lakehouse_gc/policy.yaml"

# 1. Can I read the jobs tables? This probes column names only.
access = check_access(spark, POLICY_PATH, scanners=["jobs"])
print(access.summary())

In [ ]:
# 2. What will be checked, with the settings from rules.jobs in my policy?
display(rules_dataframe(spark, POLICY_PATH, scanners=["jobs"]))

In [ ]:
# 3. Scan. Findings come most severe first: failing streaks (high) before never-run (medium).
result = run(POLICY_PATH, scanners=["jobs"], spark=spark)
print(result.summary())
display(result.to_dataframe(spark))

In [ ]:
# 4. Only the jobs that need attention.
flagged = [r for r in result.to_rows() if r["recommendation"] != "keep"]
for row in flagged:
    print(f"{row['severity']:<6} {row['rule']:<20} {row['resource']}")
    print(f"       {row['reasons']}")

In [ ]:
# 5. Try other thresholds without editing the file: stricter streak, younger jobs,
#    one workspace only (workspace_ids: [] means every workspace the tables show).
from lakehouse_gc.core import loads_policy

what_if = loads_policy("""
policy_version: 1
rules:
  jobs:
    never_run_days: 14
    failing_days: 2
    failure_states: ["FAILED", "ERROR", "TIMED_OUT"]
    workspace_ids: []
""")
print(run(policy=what_if, scanners=["jobs"], spark=spark).summary())